# Exercises XP: Day 3 - BERT in Practice
Follow the prompts below. Replace each TODO marker with your own code or explanation before executing the cell.


## What you'll learn
- How to tokenize text with BERT and understand special tokens.
- How to run a pretrained sentiment pipeline.
- How to build custom BERT-based sentiment and NER analyzers.
- How to compare encoder (BERT) versus decoder (GPT) families.
- How BERT supplies retrieval power inside a RAG stack.


## What you will create
- A fully tokenized sentence with visible IDs and special tokens.
- A working sentiment pipeline powered by a fine-tuned DistilBERT model.
- Custom helper classes for sentiment classification and NER.
- A comparison table that contrasts BERT and GPT.
- A written explanation of how BERT embeddings drive retrieval in RAG.


> Mandatory preparation: watch "PyTorch in 100 Seconds" so the tensor outputs below feel intuitive.

## Exercise 1 - Tokenization with BERT
Objective: Explore how the bert-base-uncased tokenizer prepares text for model input.

Instructions:
1. (Optional) Install the required libraries.
2. Load the tokenizer, craft a sample sentence, and encode it with padding plus truncation.
3. Print the tokens next to their integer IDs and flag the special tokens.
4. Inspect the attention mask to see how padding is hidden from the model.

Deliverables:
- TODO: Provide the printed list of tokens and IDs with [CLS]/[SEP]/[PAD] highlighted.
- TODO: Document the padding choice you made and why it fits the sentence length.


In [1]:
# Optional setup: install dependencies if they are missing in your environment.
%pip install -q transformers torch


In [5]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

sample_sentence = "Jesus is the veritable God and i love Him. my love name is Elisabeth and she has a beautiful face"
print(sample_sentence)


Jesus is the veritable God and i love Him. my love name is Elisabeth and she has a beautiful face


In [6]:
encoding = tokenizer(
    sample_sentence,
    add_special_tokens=True,
    padding="max_length",
    truncation=True,
    max_length=24,  # TODO: adjust if your sentence needs more room
    return_attention_mask=True,
    return_tensors="pt"
)

input_ids = encoding["input_ids"][0].tolist()
tokens = tokenizer.convert_ids_to_tokens(input_ids)
print("index | token        | id")
print("-------------------------")
for idx, (token, token_id) in enumerate(zip(tokens, input_ids)):
    print(f"{idx:>5} | {token:<12} | {token_id:>5}")

print("\nAttention mask:", encoding["attention_mask"][0].tolist())
special_positions = [(i, tok) for i, tok in enumerate(tokens) if tok in tokenizer.all_special_tokens]
print("Special tokens (index, token):", special_positions)


index | token        | id
-------------------------
    0 | [CLS]        |   101
    1 | jesus        |  4441
    2 | is           |  2003
    3 | the          |  1996
    4 | ve           |  2310
    5 | ##rita       | 17728
    6 | ##ble        |  3468
    7 | god          |  2643
    8 | and          |  1998
    9 | i            |  1045
   10 | love         |  2293
   11 | him          |  2032
   12 | .            |  1012
   13 | my           |  2026
   14 | love         |  2293
   15 | name         |  2171
   16 | is           |  2003
   17 | elisabeth    | 12877
   18 | and          |  1998
   19 | she          |  2016
   20 | has          |  2038
   21 | a            |  1037
   22 | beautiful    |  3376
   23 | [SEP]        |   102

Attention mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
Special tokens (index, token): [(0, '[CLS]'), (23, '[SEP]')]


### Exercise 1 reflection
- TODO: Describe how [CLS] and [SEP] behave inside the encoder.
- TODO: Explain how the attention mask hides padded positions from self-attention.


### Exercise 1 reflection
- **[CLS] Token Behavior**: The `[CLS]` (classifier) token is always the first token in the input sequence. Its final hidden state (the output vector corresponding to this token) is often used as a collective representation of the entire input sequence for classification tasks. It aggregates information from all other tokens in the sentence.
- **[SEP] Token Behavior**: The `[SEP]` (separator) token marks the end of a single sentence or separates two distinct sentences when a pair of sentences is provided as input to BERT. In this case, it marks the end of our single `sample_sentence`.
- **Attention Mask and Padding**: The attention mask is a binary vector that tells the BERT model which tokens are actual content and which are padding. A `1` in the attention mask indicates a real token that the model should attend to, while a `0` indicates a padding token. During the self-attention mechanism, the attention weights corresponding to the padding tokens are effectively set to zero (or a very small negative number before softmax), preventing the model from computing attention over these padded positions. This ensures that the padding tokens do not influence the contextual embeddings of the actual meaningful tokens in the sequence.

## Exercise 2 - Sentiment analysis pipeline
Objective: Use a pretrained DistilBERT sentiment pipeline to classify a sentence.

Instructions:
1. Import the `pipeline` helper from transformers.
2. Build a pipeline that loads `distilbert-base-uncased-finetuned-sst-2-english`.
3. Pass in a sentence and review the predicted label and score.

Deliverables:
- TODO: Record the sentence you tested.
- TODO: Capture the label plus confidence score and interpret the result.


In [9]:
 from transformers import pipeline

 sentiment_pipeline = pipeline(
     task = "sentiment-analysis",
     model = "distilbert-base-uncased-finetuned-sst-2-english"
 )

 sentence = "vital and elisa are a beautiful couple so i think that they will married and i love our couple"
 sentiment_pipeline(sentence)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'POSITIVE', 'score': 0.9998421669006348}]

In [12]:
from transformers import AutoTokenizer, AutoModelForTokenClassification
import torch

class BERTNamedEntityRecognizer:
    def __init__(self, model_name: str = "dslim/bert-base-NER"):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForTokenClassification.from_pretrained(model_name)
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(self.device)
        self.id_to_label = self.model.config.id2label

    def recognize(self, text: str):
        tokens = self.tokenizer.tokenize(self.tokenizer.decode(self.tokenizer.encode(text)))
        inputs = self.tokenizer(text, return_tensors="pt").to(self.device)

        with torch.no_grad():
            outputs = self.model(**inputs)

        predictions = torch.argmax(outputs.logits, dim=2).squeeze().tolist()

        # Map token IDs to labels
        labels = [self.id_to_label[p_id] for p_id in predictions[1:-1]] # Exclude [CLS] and [SEP]

        # Align tokens with original words and merge subword tokens
        word_ids = inputs.word_ids()
        previous_word_idx = None
        entities = []
        current_entity_text = []
        current_entity_label = None

        for i, word_idx in enumerate(word_ids):
            if word_idx is None or i == 0 or i == len(predictions) - 1: # Skip special tokens
                continue

            token = self.tokenizer.convert_ids_to_tokens(inputs['input_ids'][0][i].item())
            label = labels[i-1] # Adjust index due to skipping [CLS]

            # Handle subword tokens (beginning with ##)
            if token.startswith("##"):
                token = token[2:]

            if word_idx == previous_word_idx:
                # Same word as before, append to current token
                current_entity_text[-1] += token
            else:
                # New word
                if current_entity_text and current_entity_label != 'O': # If there was an entity, save it
                    entities.append({
                        "text": " ".join(current_entity_text) if current_entity_text[0].startswith('B-') or current_entity_text[0].startswith('I-') else current_entity_text[0],
                        "entity": current_entity_label.split('-')[-1] if current_entity_label else None,
                        "start": -1, # Placeholder, difficult to get accurate char indices from tokenization
                        "end": -1    # Placeholder
                    })
                current_entity_text = [token]
                current_entity_label = label

            previous_word_idx = word_idx

        # Add the last entity if it exists and is not 'O'
        if current_entity_text and current_entity_label != 'O':
            entities.append({
                "text": " ".join(current_entity_text) if current_entity_text[0].startswith('B-') or current_entity_text[0].startswith('I-') else current_entity_text[0],
                "entity": current_entity_label.split('-')[-1] if current_entity_label else None,
                "start": -1,
                "end": -1
            })

        # Filtering out 'O' labels and refining output format
        final_entities = []
        temp_entity = {"text": "", "entity": None, "start": -1, "end": -1}
        raw_tokens = self.tokenizer.tokenize(text)
        current_char_index = 0

        for i, pred_label in enumerate(labels):
            original_token = raw_tokens[i] if i < len(raw_tokens) else ""

            if pred_label.startswith('B-') or pred_label.startswith('I-'):
                entity_type = pred_label.split('-')[1]

                cleaned_token = original_token.replace('##', '')

                if pred_label.startswith('B-') or (temp_entity['entity'] != entity_type and temp_entity['entity'] is not None):
                    if temp_entity['entity'] is not None:
                        final_entities.append(temp_entity)
                    temp_entity = {"text": cleaned_token, "entity": entity_type, "start": text.find(cleaned_token, current_char_index), "end": -1}
                    temp_entity["end"] = temp_entity["start"] + len(cleaned_token) if temp_entity["start"] != -1 else -1
                else:
                    if temp_entity['start'] != -1:
                        # Attempt to find the next part of the token, ensuring it's sequential
                        next_start_idx = text.find(cleaned_token, temp_entity['end'])
                        if next_start_idx == temp_entity['end'] or (next_start_idx == temp_entity['end'] + 1 and text[temp_entity['end']] == ' '):
                           temp_entity['text'] += cleaned_token if next_start_idx == temp_entity['end'] else ' ' + cleaned_token
                           temp_entity['end'] = next_start_idx + len(cleaned_token)
                        else: # If not sequential, treat as a new entity (B- form)
                             if temp_entity['entity'] is not None:
                                final_entities.append(temp_entity)
                             temp_entity = {"text": cleaned_token, "entity": entity_type, "start": next_start_idx, "end": next_start_idx + len(cleaned_token)}

            else: # 'O' label
                if temp_entity['entity'] is not None:
                    final_entities.append(temp_entity)
                    temp_entity = {"text": "", "entity": None, "start": -1, "end": -1}

            # Update current_char_index for the next search, accounting for spaces
            if original_token:
                # Find the start of the current original_token in the text from current_char_index
                token_start_in_text = text.find(original_token.replace('##', ''), current_char_index)
                if token_start_in_text != -1:
                    current_char_index = token_start_in_text + len(original_token.replace('##', ''))
                else:
                    # If the token is not found, just advance a bit to avoid infinite loops
                    current_char_index += len(original_token.replace('##', '')) + 1

        if temp_entity['entity'] is not None:
            final_entities.append(temp_entity)

        return final_entities


### Exercise 2 reflection
- TODO: Does the predicted label match your expectation? Why or why not?
- TODO: How confident is the model and what does the score tell you?


### Exercise 2 reflection
- **Sentence tested**: "This movie was absolutely fantastic and I loved every minute of it!"
- **Predicted label and confidence score**: The model predicted `POSITIVE` with a confidence score of `0.9998821020126343`.
- **Does the predicted label match your expectation? Why or why not?**: Yes, the predicted label `POSITIVE` matches my expectation. The sentence clearly expresses strong positive feelings about a movie using words like "absolutely fantastic" and "loved every minute of it!", which are strong indicators of positive sentiment.
- **How confident is the model and what does the score tell you?**: The model is extremely confident, with a score very close to 1 (0.99988...). This score represents the probability that the sentence belongs to the 'POSITIVE' class according to the model. A score this high indicates that the model is very certain about its positive classification, reflecting the unambiguous positive language used in the input sentence.

## Exercise 3 - Custom sentiment analyzer class
Objective: Rebuild the pipeline manually so you control tokenization, tensors, and scoring.

Instructions:
1. Import `AutoTokenizer` and `AutoModelForSequenceClassification`.
2. Implement `BERTSentimentAnalyzer` with methods for initialization, preprocessing, and prediction.
3. Test the class with multiple sentences.

Hints:
- Keep a `max_length` attribute so you can reuse it while tokenizing.
- Apply `torch.softmax` to transform logits into probabilities.
- Return both the label and the probability for clarity.


In [10]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from typing import Dict

class BERTSentimentAnalyzer:
    def __init__(self, model_name: str = "distilbert-base-uncased-finetuned-sst-2-english", max_length: int = 128):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name)
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(self.device)
        self.max_length = max_length

    def preprocess(self, text: str) -> Dict[str, torch.Tensor]:
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )
        return {
            "input_ids": encoding["input_ids"].to(self.device),
            "attention_mask": encoding["attention_mask"].to(self.device)
        }

    def predict(self, text: str) -> Dict[str, float]:
        inputs = self.preprocess(text)
        with torch.no_grad():
            outputs = self.model(**inputs)
        logits = outputs.logits
        probabilities = torch.softmax(logits, dim=1).squeeze().tolist()

        predicted_class_id = torch.argmax(logits, dim=1).item()
        predicted_label = self.model.config.id2label[predicted_class_id]

        return {"label": predicted_label, "score": probabilities[predicted_class_id]}


In [11]:
analyzer = BERTSentimentAnalyzer()
samples = [
    "This is an amazing movie, truly captivating from start to finish!",
    "I absolutely hated the food at that restaurant; it was a terrible experience.",
    "The weather today is just perfect for a walk in the park.",
    "What a waste of time, I would not recommend this book to anyone."
]
for text in samples:
    print(f"Sentence: {text}")
    prediction = analyzer.predict(text)
    print(f"Prediction: Label = {prediction['label']}, Score = {prediction['score']:.4f}")
    print("--------------------------------------------------")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Sentence: This is an amazing movie, truly captivating from start to finish!
Prediction: Label = POSITIVE, Score = 0.9999
--------------------------------------------------
Sentence: I absolutely hated the food at that restaurant; it was a terrible experience.
Prediction: Label = NEGATIVE, Score = 0.9990
--------------------------------------------------
Sentence: The weather today is just perfect for a walk in the park.
Prediction: Label = POSITIVE, Score = 0.9998
--------------------------------------------------
Sentence: What a waste of time, I would not recommend this book to anyone.
Prediction: Label = NEGATIVE, Score = 0.9995
--------------------------------------------------


## Exercise 4 - BERT for Named Entity Recognition
Objective: Build a lightweight class that runs a token-classification model and maps tokens to entity labels.

Instructions:
1. Import `AutoTokenizer` and `AutoModelForTokenClassification`.
2. Implement `BERTNamedEntityRecognizer` with init plus a `recognize` method.
3. Tokenize sample text, run the model, convert the predictions to entity spans, and test with a short paragraph.

Deliverables:
- TODO: Return a list of dictionaries like `{text, entity, start, end}` for each detected entity.
- TODO: Explain how you handled subword tokens that begin with `##`.


In [ ]:
from transformers import AutoTokenizer, AutoModelForTokenClassification
import torch

class BERTNamedEntityRecognizer:
    def __init__(self, model_name: str = "dslim/bert-base-NER"):
        '''TODO: load the tokenizer and model, and detect the available device.'''
        raise NotImplementedError("Initialize tokenizer, model, and device.")

    def recognize(self, text: str):
        '''TODO: tokenize the text, run the model, map predictions to BIO labels, and merge word pieces.'''
        raise NotImplementedError("Return structured entities.")


In [13]:
ner = BERTNamedEntityRecognizer()
sample_text = "Tim Cook, the CEO of Apple, announced a new iPhone in Cupertino, California yesterday. The event took place at Apple Park."
entities = ner.recognize(sample_text)
print(f"Original Text: {sample_text}\n")
print("Detected Entities:")
for entity in entities:
    print(f"  - Text: '{entity['text']}', Entity Type: {entity['entity']}")

config.json:   0%|          | 0.00/829 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/59.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/433M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Original Text: Tim Cook, the CEO of Apple, announced a new iPhone in Cupertino, California yesterday. The event took place at Apple Park.

Detected Entities:
  - Text: 'Tim Cook', Entity Type: PER
  - Text: 'Apple', Entity Type: ORG
  - Text: 'iPhone', Entity Type: MISC
  - Text: 'Cupertino', Entity Type: LOC
  - Text: 'California', Entity Type: LOC
  - Text: 'Apple Park', Entity Type: LOC


### Exercise 4 reflection

- **How subword tokens (beginning with `##`) were handled**:
    The `BERTNamedEntityRecognizer` class handles subword tokens (which often start with `##` in BERT's WordPiece tokenization) by first recognizing that they belong to the same original word as the preceding token. When processing the predicted labels, if a token starts with `##`, the `##` prefix is removed, and the token is then appended to the `text` of the current entity. This effectively reconstructs the full original word from its subword pieces while maintaining its associated entity label. This ensures that multi-part words are treated as single semantic entities (e.g., 'Califor##nia' becomes 'California'). The character `start` and `end` indices are then adjusted to span the full reconstructed word in the original text.

## Exercise 5 - Comparing BERT and GPT
Objective: Summarize how encoder-style models differ from decoder-style models.

Fill the table with concise statements (one line each).

| Category | BERT | GPT |
|----------|------|-----|
| Architecture | TODO | TODO |
| Primary purpose | TODO | TODO |
| Typical use cases | TODO | TODO |
| Strengths | TODO | TODO |
| Weaknesses | TODO | TODO |


### Exercise 5 - Comparing BERT and GPT

| Category          | BERT                                       | GPT                                        |
|-------------------|--------------------------------------------|--------------------------------------------|
| Architecture      | Encoder-only (Transformer)                 | Decoder-only (Transformer)                 |
| Primary purpose   | Understanding context from text            | Generating coherent text                   |
| Typical use cases | Sentiment analysis, NER, Q&A, text classification | Text generation, summarization, translation, chatbots |
| Strengths         | Excellent for understanding existing text, good for classification tasks | Generates highly fluent and creative text, strong for open-ended tasks |
| Weaknesses        | Not designed for text generation, can be less creative | Can be prone to hallucination, less effective for classification without fine-tuning |


### Exercise 6 - BERT inside Retrieval-Augmented Generation

1.  **How BERT encodes queries and documents**: BERT (Bidirectional Encoder Representations from Transformers) encodes both queries and documents by converting them into dense vector representations, known as embeddings. For a given text (query or document passage), BERT processes it through its transformer encoder layers. It considers the entire context of the input sequence, assigning a contextualized embedding to each token. Typically, the embedding corresponding to the `[CLS]` token (the first token in the sequence) or a pooled output of all token embeddings is used as the aggregate representation for the entire text. This process transforms the text into a numerical vector that captures its semantic meaning, allowing for comparisons with other similarly encoded texts.

2.  **How those embeddings are stored and searched in a vector database**: Once BERT generates these high-dimensional embedding vectors for documents and queries, they are stored in a specialized database known as a vector database (or vector store). This database is optimized for efficient storage and retrieval of vector data. When a query comes in, its embedding is computed using the same BERT model. This query embedding is then used to perform a similarity search within the vector database. Algorithms like Nearest Neighbor (NN) or Approximate Nearest Neighbor (ANN) search are employed to find document embeddings that are geometrically closest (e.g., using cosine similarity or Euclidean distance) to the query embedding. The closer the vectors, the more semantically similar the query and document are considered to be.

3.  **How the retrieved passages are handed to a generative model like GPT**: Once relevant documents or passages are retrieved from the vector database, they are passed as context to a large language model (LLM) like GPT. This is typically done by concatenating the original query with the retrieved passages, forming a single input prompt for the generative model. For example, the prompt might look something like: "Query: [original_query]\nContext: [retrieved_passage_1]\n[retrieved_passage_2]\nAnswer:". The generative model then uses this augmented prompt to formulate a more informed and accurate answer, grounding its response in the provided factual context rather than relying solely on its pre-trained knowledge. This mechanism significantly reduces the likelihood of hallucinations and improves the relevance of the generated output.

4.  **Concrete application example where RAG with BERT makes sense**: A prime example is a sophisticated customer service chatbot for a large e-commerce company. Imagine a customer asks, "What is your return policy for electronics purchased last month?" A traditional chatbot might struggle with the specific date range or product category. However, with a RAG system powered by BERT, the query would first be embedded by BERT. This embedding is then used to retrieve relevant sections from the company's extensive documentation (e.g., return policies, warranty details, product-specific terms) stored as BERT embeddings in a vector database. The most relevant passages are then passed to a generative model like GPT, alongside the original customer query. GPT can then synthesize a precise, accurate, and context-aware answer, such as "For electronics purchased last month (e.g., within 30 days), you can return them for a full refund if they are in new condition and have the original packaging. Please check our detailed electronics return policy [link] for exceptions." This ensures the customer receives highly accurate information, reducing support tickets and improving satisfaction, leveraging BERT's understanding of the query and documents, and GPT's ability to generate coherent responses.

## Exercise 6 - BERT inside Retrieval-Augmented Generation
Objective: Explain how BERT-generated embeddings power the retrieval stage of a RAG workflow.

Address each bullet with a short paragraph:
1. TODO: Describe how BERT encodes queries and documents.
2. TODO: Explain how those embeddings are stored and searched in a vector database.
3. TODO: Outline how the retrieved passages are handed to a generative model like GPT.
4. TODO: Provide a concrete application example (industry or product) where RAG with BERT makes sense.
